In [2]:
# ============================================================
# FEATURE ENGINEERING & ENCODING
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

# Scikit-learn tools for encoding and scaling
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# ============================================================
# LOAD CLEANED DATASET
# ============================================================

file_path = "../data/processed/cleaned_data.csv"

df = pd.read_csv(
    file_path,
    parse_dates=["VisitDate"]
)

print("Dataset loaded successfully.")

print("\nDataset shape:")
print(df.shape)

print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully.

Dataset shape:
(52930, 22)

First 5 rows:


,TransactionId,UserId,VisitDate,VisitYear,VisitMonth,VisitModeId,VisitMode,AttractionId,Attraction,AttractionAddress,...,AttractionType,Rating,CityId,CityName,CountryId,Country,RegionId,Region,ContinentId,Continent
0,3,70456,2022-10-01,2022,10,2,Couples,640,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",...,Nature & Wildlife Areas,5,4341.0,Guildford,163,United Kingdom,21,Western Europe,5,Europe
1,8,7567,2022-10-01,2022,10,4,Friends,640,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",...,Nature & Wildlife Areas,5,464.0,Ontario,48,Canada,8,Northern America,2,America
2,9,79069,2022-10-01,2022,10,3,Family,640,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",...,Nature & Wildlife Areas,5,774.0,Brazil,54,Brazil,9,South America,2,America
3,10,31019,2022-10-01,2022,10,3,Family,640,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",...,Nature & Wildlife Areas,3,583.0,Zurich,135,Switzerland,17,Central Europe,5,Europe
4,15,43611,2022-10-01,2022,10,2,Couples,640,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",...,Nature & Wildlife Areas,3,1396.0,Manchester,163,United Kingdom,21,Western Europe,5,Europe


In [5]:
# ============================================================
# CHECK AVAILABLE COLUMNS
# ============================================================

print("Columns in the dataset:\n")

for i, column in enumerate(df.columns, start=1):
    print(f"{i:2}. {column}")

Columns in the dataset:

 1. TransactionId
 2. UserId
 3. VisitDate
 4. VisitYear
 5. VisitMonth
 6. VisitModeId
 7. VisitMode
 8. AttractionId
 9. Attraction
10. AttractionAddress
11. AttractionCityId
12. AttractionTypeId
13. AttractionType
14. Rating
15. CityId
16. CityName
17. CountryId
18. Country
19. RegionId
20. Region
21. ContinentId
22. Continent


## Create user-level aggregate features

In [4]:
# ============================================================
# USER-LEVEL AGGREGATE FEATURES
# ============================================================

# ------------------------------------------------------------
# 1. Number of visits made by each user
# ------------------------------------------------------------

user_visit_count = (
    df.groupby("UserId")
      .size()
      .reset_index(name="UserVisitCount")
)


# ------------------------------------------------------------
# 2. Average rating given by each user
# ------------------------------------------------------------

user_avg_rating = (
    df.groupby("UserId")["Rating"]
      .mean()
      .reset_index(name="UserAverageRating")
)


# ------------------------------------------------------------
# 3. Most common visit mode for each user
# ------------------------------------------------------------

def most_common_value(series):
    """
    Return the most frequently occurring value.

    If there is a tie, pandas returns the first value.
    """

    mode_values = series.mode()

    if len(mode_values) > 0:
        return mode_values.iloc[0]

    return pd.NA


user_common_mode = (
    df.groupby("UserId")["VisitMode"]
      .agg(most_common_value)
      .reset_index(name="UserMostCommonVisitMode")
)


# ------------------------------------------------------------
# Combine all user-level features
# ------------------------------------------------------------

user_features = (
    user_visit_count
    .merge(user_avg_rating, on="UserId")
    .merge(user_common_mode, on="UserId")
)


print("User-level features:")
display(user_features.head(10))

User-level features:


,UserId,UserVisitCount,UserAverageRating,UserMostCommonVisitMode
0,14,3,4.666667,Friends
1,16,10,4.700000,Family
2,20,1,4.000000,Family
3,23,1,5.000000,Friends
4,25,1,5.000000,Friends
5,26,1,5.000000,Couples
6,27,1,4.000000,Couples
7,28,1,5.000000,Couples
8,29,1,5.000000,Couples
9,32,1,4.000000,Family


## Create attraction-level aggregate features

In [5]:
# ============================================================
# ATTRACTION-LEVEL AGGREGATE FEATURES
# ============================================================

# ------------------------------------------------------------
# 1. Number of visits received by each attraction
# ------------------------------------------------------------

attraction_visit_count = (
    df.groupby("AttractionId")
      .size()
      .reset_index(name="AttractionVisitCount")
)


# ------------------------------------------------------------
# 2. Average rating received by each attraction
# ------------------------------------------------------------

attraction_avg_rating = (
    df.groupby("AttractionId")["Rating"]
      .mean()
      .reset_index(name="AttractionAverageRating")
)


# ------------------------------------------------------------
# Combine attraction features
# ------------------------------------------------------------

attraction_features = (
    attraction_visit_count
    .merge(
        attraction_avg_rating,
        on="AttractionId"
    )
)


print("Attraction-level features:")
display(attraction_features)

Attraction-level features:


,AttractionId,AttractionVisitCount,AttractionAverageRating
0,369,2765,3.415190
1,481,2104,4.275665
2,640,13198,4.267086
3,650,3044,3.976347
4,673,2914,3.800618
5,737,3352,4.194809
6,748,5815,4.157524
7,749,2190,3.895434
8,824,3359,4.219411
9,841,6429,4.646601


## Create useful non-leaking aggregate features

In [6]:
# ============================================================
# MERGE SAFE AGGREGATE FEATURES
# ============================================================

# Keep only aggregates that do not directly use the prediction target
safe_user_features = user_features[
    [
        "UserId",
        "UserVisitCount"
    ]
]

safe_attraction_features = attraction_features[
    [
        "AttractionId",
        "AttractionVisitCount"
    ]
]


# ------------------------------------------------------------
# Merge user-level feature
# ------------------------------------------------------------

df_features = df.merge(
    safe_user_features,
    on="UserId",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# Merge attraction-level feature
# ------------------------------------------------------------

df_features = df_features.merge(
    safe_attraction_features,
    on="AttractionId",
    how="left",
    validate="many_to_one"
)


print("Dataset after adding aggregate features:")
print(df_features.shape)

display(
    df_features[
        [
            "UserId",
            "UserVisitCount",
            "AttractionId",
            "AttractionVisitCount"
        ]
    ].head(10)
)

Dataset after adding aggregate features:
(52930, 24)


,UserId,UserVisitCount,AttractionId,AttractionVisitCount
0,70456,1,640,13198
1,7567,1,640,13198
2,79069,1,640,13198
3,31019,2,640,13198
4,43611,3,640,13198
5,43471,2,640,13198
6,76492,2,640,13198
7,20977,1,640,13198
8,18655,1,640,13198
9,62907,14,640,13198


## Add the requested aggregate features for reference

In [7]:
# ============================================================
# REQUESTED TARGET-BASED AGGREGATES
# ============================================================

print("User average rating:")
display(user_avg_rating.head())

print("\nUser most common visit mode:")
display(user_common_mode.head())

print("\nAttraction average rating:")
display(attraction_avg_rating.head())

User average rating:


,UserId,UserAverageRating
0,14,4.666667
1,16,4.700000
2,20,4.000000
3,23,5.000000
4,25,5.000000



User most common visit mode:


,UserId,UserMostCommonVisitMode
0,14,Friends
1,16,Family
2,20,Family
3,23,Friends
4,25,Friends



Attraction average rating:


,AttractionId,AttractionAverageRating
0,369,3.415190
1,481,4.275665
2,640,4.267086
3,650,3.976347
4,673,3.800618


## Decide which columns are categorical

In [8]:
# ============================================================
# DEFINE TARGETS
# ============================================================

REGRESSION_TARGET = "Rating"

CLASSIFICATION_TARGET = "VisitMode"

print("Regression target:", REGRESSION_TARGET)
print("Classification target:", CLASSIFICATION_TARGET)

Regression target: Rating
Classification target: VisitMode


## Remove columns that shouldn't be model features

In [9]:
# ============================================================
# DEFINE COMMON FEATURES
# ============================================================

columns_to_remove = [
    "TransactionId",
    "AttractionAddress",

    # Raw IDs are identifiers rather than meaningful numerical features
    "UserId",
    "AttractionId",
    "CityId",
    "CountryId",
    "RegionId",
    "ContinentId",

    # This is redundant with VisitMode
    "VisitModeId",

    # This is redundant with AttractionType
    "AttractionTypeId",

    # AttractionCityId has a known source-data ID inconsistency
    "AttractionCityId"
]

print("Columns excluded from model features:")

for column in columns_to_remove:
    print("-", column)

Columns excluded from model features:
- TransactionId
- AttractionAddress
- UserId
- AttractionId
- CityId
- CountryId
- RegionId
- ContinentId
- VisitModeId
- AttractionTypeId
- AttractionCityId


## Create the regression dataset

In [10]:
# ============================================================
# CREATE REGRESSION DATASET
# ============================================================

regression_df = df_features.copy()


# Remove columns that should not be model features
regression_features = regression_df.drop(
    columns=columns_to_remove + ["Rating"],
    errors="ignore"
)


# Add target at the end
regression_features["Rating"] = regression_df["Rating"]


print("Regression dataset shape:")
print(regression_features.shape)

display(regression_features.head())

Regression dataset shape:
(52930, 13)


,VisitDate,VisitYear,VisitMonth,VisitMode,Attraction,AttractionType,CityName,Country,Region,Continent,UserVisitCount,AttractionVisitCount,Rating
0,2022-10-01,2022,10,Couples,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Guildford,United Kingdom,Western Europe,Europe,1,13198,5
1,2022-10-01,2022,10,Friends,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Ontario,Canada,Northern America,America,1,13198,5
2,2022-10-01,2022,10,Family,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Brazil,Brazil,South America,America,1,13198,5
3,2022-10-01,2022,10,Family,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Zurich,Switzerland,Central Europe,Europe,2,13198,3
4,2022-10-01,2022,10,Couples,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Manchester,United Kingdom,Western Europe,Europe,3,13198,3


## Create the classification dataset

In [11]:
# ============================================================
# CREATE CLASSIFICATION DATASET
# ============================================================

classification_df = df_features.copy()


# Remove model-excluded columns
classification_features = classification_df.drop(
    columns=columns_to_remove + ["VisitMode"],
    errors="ignore"
)


# Add target at the end
classification_features["VisitMode"] = classification_df["VisitMode"]


print("Classification dataset shape:")
print(classification_features.shape)

display(classification_features.head())

Classification dataset shape:
(52930, 13)


,VisitDate,VisitYear,VisitMonth,Attraction,AttractionType,Rating,CityName,Country,Region,Continent,UserVisitCount,AttractionVisitCount,VisitMode
0,2022-10-01,2022,10,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,5,Guildford,United Kingdom,Western Europe,Europe,1,13198,Couples
1,2022-10-01,2022,10,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,5,Ontario,Canada,Northern America,America,1,13198,Friends
2,2022-10-01,2022,10,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,5,Brazil,Brazil,South America,America,1,13198,Family
3,2022-10-01,2022,10,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,3,Zurich,Switzerland,Central Europe,Europe,2,13198,Family
4,2022-10-01,2022,10,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,3,Manchester,United Kingdom,Western Europe,Europe,3,13198,Couples


## Encode categorical variables

In [12]:
# ============================================================
# IDENTIFY CATEGORICAL COLUMNS
# ============================================================

categorical_columns = [
    "VisitMode",
    "Continent",
    "Country",
    "Region",
    "CityName",
    "Attraction",
    "AttractionType"
]

# Keep only columns that exist
categorical_columns = [
    column
    for column in categorical_columns
    if column in regression_features.columns
]

print("Categorical columns:")
print(categorical_columns)

Categorical columns:
['VisitMode', 'Continent', 'Country', 'Region', 'CityName', 'Attraction', 'AttractionType']


## One-hot and Frequency encoding regression features

In [13]:
# ============================================================
# STEP 14 - MEMORY-SAFE CATEGORICAL ENCODING
# ============================================================

# Make copies so the original data is not modified
regression_encoded = regression_features.copy()
classification_encoded = classification_features.copy()


# ------------------------------------------------------------
# 1. Frequency encoding
# ------------------------------------------------------------
# Frequency encoding is useful for high-cardinality columns
# such as CityName and Country.
#
# Instead of creating thousands of columns, we create
# ONE numeric column representing how frequently each
# category occurs in the dataset.
# ------------------------------------------------------------

def add_frequency_encoding(data, column):
    """
    Replace a categorical column with its frequency.

    Example:
    If Hyderabad appears 5,000 times,
    Hyderabad -> 5000 / total_rows
    """

    frequency = data[column].value_counts(normalize=True)

    data[column + "_Frequency"] = (
        data[column]
        .fillna("Unknown")
        .map(frequency)
        .fillna(0)
    )

    # Remove original text column
    data.drop(columns=[column], inplace=True)

    return data


# High-cardinality columns
high_cardinality_columns = [
    "Country",
    "CityName"
]


# Apply frequency encoding
for column in high_cardinality_columns:

    if column in regression_encoded.columns:
        regression_encoded = add_frequency_encoding(
            regression_encoded,
            column
        )

    if column in classification_encoded.columns:
        classification_encoded = add_frequency_encoding(
            classification_encoded,
            column
        )


# ------------------------------------------------------------
# 2. One-hot encode LOW-cardinality columns
# ------------------------------------------------------------

regression_low_cardinality = [
    "VisitMode",
    "Continent",
    "Region",
    "AttractionType",
    "Attraction"
]

classification_low_cardinality = [
    "Continent",
    "Region",
    "AttractionType",
    "Attraction"
]


# Fill missing categorical values
for column in regression_low_cardinality:

    if column in regression_encoded.columns:
        regression_encoded[column] = (
            regression_encoded[column]
            .fillna("Unknown")
        )


for column in classification_low_cardinality:

    if column in classification_encoded.columns:
        classification_encoded[column] = (
            classification_encoded[column]
            .fillna("Unknown")
        )


# ------------------------------------------------------------
# 3. Create dummy variables
# ------------------------------------------------------------

regression_encoded = pd.get_dummies(
    regression_encoded,
    columns=regression_low_cardinality,
    dtype="int8"
)

classification_encoded = pd.get_dummies(
    classification_encoded,
    columns=classification_low_cardinality,
    dtype="int8"
)


# ------------------------------------------------------------
# 4. Check the result
# ------------------------------------------------------------

print("Regression dataset shape:")
print(regression_encoded.shape)

print("\nClassification dataset shape:")
print(classification_encoded.shape)

print("\nRegression memory usage:")
print(
    round(
        regression_encoded.memory_usage(deep=True).sum() / (1024 ** 2),
        2
    ),
    "MB"
)

print("\nClassification memory usage:")
print(
    round(
        classification_encoded.memory_usage(deep=True).sum() / (1024 ** 2),
        2
    ),
    "MB"
)

Regression dataset shape:
(52930, 87)

Classification dataset shape:
(52930, 83)

Regression memory usage:
7.22 MB

Classification memory usage:
7.7 MB


### Check the encoded datasets

In [14]:
# Check the final shapes
print("Regression dataset shape:", regression_encoded.shape)
print("Classification dataset shape:", classification_encoded.shape)

# Check data types
print("\nRegression data types:")
print(regression_encoded.dtypes.value_counts())

print("\nClassification data types:")
print(classification_encoded.dtypes.value_counts())

Regression dataset shape: (52930, 87)
Classification dataset shape: (52930, 83)

Regression data types:
int8              79
int64              5
float64            2
datetime64[us]     1
Name: count, dtype: int64

Classification data types:
int8              74
int64              5
float64            2
datetime64[us]     1
str                1
Name: count, dtype: int64


### check missing values

In [15]:
print("Regression missing values:",
      regression_encoded.isnull().sum().sum())

print("Classification missing values:",
      classification_encoded.isnull().sum().sum())

Regression missing values: 0
Classification missing values: 0


### check the infinite values

In [16]:
print(
    "Regression infinite values:",
    np.isinf(
        regression_encoded.select_dtypes(include=np.number)
    ).sum().sum()
)

print(
    "Classification infinite values:",
    np.isinf(
        classification_encoded.select_dtypes(include=np.number)
    ).sum().sum()
)

Regression infinite values: 0
Classification infinite values: 0


### Make sure the target columns are correct

In [17]:
print("Last regression column:",
      regression_encoded.columns[-1])

print("Last classification column:",
      classification_encoded.columns[-1])

print("\nRating values:")
print(regression_encoded["Rating"].value_counts().sort_index())

print("\nVisitMode values:")
print(classification_encoded["VisitMode"].value_counts().sort_index())

Last regression column: Attraction_Yogyakarta Palace
Last classification column: Attraction_Yogyakarta Palace

Rating values:
Rating
1     1263
2     2035
3     7730
4    17966
5    23936
Name: count, dtype: int64

VisitMode values:
VisitMode
Business      623
Couples     21620
Family      15217
Friends     10945
Solo         4525
Name: count, dtype: int64


### Save the two Day 3 datasets

In [18]:
regression_encoded.to_csv(
    "../data/processed/model_data_regression.csv",
    index=False
)

classification_encoded.to_csv(
    "../data/processed/model_data_classification.csv",
    index=False
)

print("Both Day 3 datasets saved successfully.")

Both Day 3 datasets saved successfully.
